# Step 13: Compare Reactome analysis results to Enrichr results

The Reactome analysis showed a really nice enrichment for the neutrophil degranulation pathway. However, this wasn't corroborated by any of the GSEA results. So, as a check, we're going to see what Enrichr gives us.

## Setup

In [1]:
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP01_DIR = "step01_outputs"
STEP01_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP01_FILE_PATH = os.path.join(STEP01_DIR, STEP01_FILE)

STEP13_DIR = "step13_outputs"
GSEAPY_DIR_PATH = os.path.join(STEP13_DIR, "gseapy")

if not os.path.isdir(STEP13_DIR):
    os.mkdir(STEP13_DIR)

In [3]:
# Read in the expression data from step 1
data = pd.read_csv(STEP01_FILE_PATH, sep='\t', index_col=0)

## Query Enrichr through GSEApy

In [57]:
# Note: Enrichr only supports the 2016 version of Reactome, and
# that doesn't have the neutrophil degranulation pathway in it.
# But when you try and use a local .gmt file, GSEApy fails to 
# calculate p values. So, we'll use the GO_Biological_Process_2018
# library.

all_enrich = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():

    sig_diff_genes = data.\
        loc[
            data["reject_null"] &
            (data["cancer_type"] == cancer_type), 
            "protein_str"
        ].\
        tolist()

    enr = gp.enrichr(
        gene_list=sig_diff_genes,
        gene_sets="GO_Biological_Process_2018",
        outdir=GSEAPY_DIR_PATH,
        no_plot=True,
        cutoff=1
    )

    res = enr.res2d
    nums = res["Overlap"].str.split("/", n=1, expand=True)
    res = res.assign(
        prop=nums[0].astype(int) / nums[1].astype(int),
        cancer_type=cancer_type
    )
    
    all_enrich = all_enrich.append(res)

In [84]:
all_enrichments = all_enrich.\
    assign(cancer_rank_pval=all_enrichments.groupby("cancer_type")["Combined Score"].rank(ascending=True)).\
    sort_values(by=["cancer_type", "cancer_rank_pval"]).\
    reset_index(drop=True)

In [85]:
enrichment_summary = all_enrichments[["Term"]].drop_duplicates(keep="first")

pathway_times_enriched = enrichment_summary["Term"].apply(
    lambda x: all_enrichments[all_enrichments["Term"] == x].shape[0])

avg_rank = enrichment_summary["Term"].apply(
    lambda x: all_enrichments.loc[all_enrichments["Term"] == x, "cancer_rank_pval"].mean())

enrichment_summary = enrichment_summary.\
    assign(
        pathway_times_enriched=pathway_times_enriched,
        pathway_avg_rank=avg_rank).\
    sort_values(
        by=["pathway_times_enriched", "pathway_avg_rank"],
        ascending=[False, True]).\
    reset_index(drop=True)

As you can see below, neutrophil degranulation gets highly ranked when we use Enrichr's combination of a Fisher's exact test and the Z-score.

In [86]:
enrichment_summary

,Term,pathway_times_enriched,pathway_avg_rank
0,ribosome biogenesis (GO:0042254),8,8.500000
1,neutrophil degranulation (GO:0043312),8,8.875000
2,gene expression (GO:0010467),8,10.125000
3,rRNA processing (GO:0006364),8,10.500000
4,neutrophil mediated immunity (GO:0002446),8,11.375000
5,ncRNA processing (GO:0034470),8,13.125000
6,translation (GO:0006412),8,13.375000
7,rRNA metabolic process (GO:0016072),8,13.500000
8,neutrophil activation involved in immune respo...,8,14.875000
9,cellular protein metabolic process (GO:0044267),8,15.000000
